# Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
import os
warnings.filterwarnings("ignore")

# sklearn utilities
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted, validate_data
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils import check_random_state
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import QuantileRegressor
from scipy.stats import norm, t, logistic, cauchy, gumbel_r, pearsonr, spearmanr
from itertools import product

# LP-based quantile regression
from sklearn.linear_model import QuantileRegressor

# MSE
from sklearn.metrics import mean_squared_error

# Statsmodels (IRLS quantile regression)
import statsmodels.api as sm

# proxSGD
from proxsgd_quantile import SGDQuantileRegressor

from scipy.stats import norm
from scipy.stats import linregress
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, LogFormatterMathtext

# Metrics

In [ ]:
# Core Metrics
def pinball_loss(y_true, y_pred, tau):
    """
    Mean pinball loss for quantile regression.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    r = y_true - y_pred
    return np.mean(np.maximum(tau * r, (tau - 1) * r))


def rmse(y_true, y_pred):
    """
    Root mean squared error.
    """
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def coverage_rate(y_true, y_lower, y_upper):
    """
    Empirical coverage of prediction intervals.
    """
    y_true = np.asarray(y_true)
    return np.mean((y_true >= y_lower) & (y_true <= y_upper))


def mean_interval_width(y_lower, y_upper):
    """
    Average width of prediction intervals.
    """
    return np.mean(y_upper - y_lower)


def crossing_rate(y_lower, y_upper):
    """
    Fraction of samples where lower quantile exceeds upper quantile.
    """
    return np.mean(y_lower > y_upper)


def winkler_score(y_true, y_lower, y_upper, alpha):
    """
    Winkler score for prediction intervals.
    Lower is better.
    """
    y_true = np.asarray(y_true)
    width = y_upper - y_lower

    below = y_true < y_lower
    above = y_true > y_upper

    score = width.copy()
    score[below] += (2 / alpha) * (y_lower[below] - y_true[below])
    score[above] += (2 / alpha) * (y_true[above] - y_upper[above])

    return np.mean(score)



# Interval Metric Wrappers
def interval_metrics_basic(y_true, y_lower, y_upper):
    """
    Basic interval metrics: coverage, width, crossing.
    """
    return {
        "coverage": coverage_rate(y_true, y_lower, y_upper),
        "mean_width": mean_interval_width(y_lower, y_upper),
        "crossing_rate": crossing_rate(y_lower, y_upper),
    }


def interval_metrics_with_winkler(y_true, y_lower, y_upper, alpha):
    """
    Extended interval metrics including Winkler score.
    """
    metrics = interval_metrics_basic(y_true, y_lower, y_upper)
    metrics["winkler"] = winkler_score(y_true, y_lower, y_upper, alpha)
    return metrics



# KKT Stationarity Diagnostic
def kkt_grad_inf(X, y, coef, intercept, tau):
    """
    Infinity norm of stationarity residual:
        g = (1/n) X^T s,  where s_i = tau - 1{r_i < 0}

    This is a proxy for KKT optimality (exact when l1=0).
    """
    X = np.asarray(X)
    y = np.asarray(y)

    r = y - (X @ coef + intercept)
    s = tau - (r < 0).astype(float)

    g = (X.T @ s) / X.shape[0]
    return np.linalg.norm(g, ord=np.inf)


def kkt_stationarity_qr(X, y, coef, intercept, tau):
    """
    Returns both coefficient and intercept stationarity diagnostics.
    """
    X = np.asarray(X)
    y = np.asarray(y)

    r = y - (X @ coef + intercept)
    s = tau - (r < 0).astype(float)

    g = (X.T @ s) / X.shape[0]
    g0 = np.mean(s)  # intercept condition

    return {
        "kkt_grad_inf": np.linalg.norm(g, ord=np.inf),
        "kkt_intercept_abs": np.abs(g0),
    }

# Data Loaders

## Synthetic data

In [ ]:
# ================================
# Synthetic Data Generators
# ================================

from scipy.stats import norm, t, logistic, cauchy, gumbel_r


def sample_design_X(n, d, rng, covariance="toeplitz", rho=0.0):
    """
    Sample Gaussian design matrix with either identity or Toeplitz covariance.
    """
    if not (0 <= rho < 1):
        raise ValueError("rho must be in [0, 1).")

    if covariance == "identity" or rho == 0.0:
        Sigma = np.eye(d)
    elif covariance == "toeplitz":
        idx = np.arange(d)
        Sigma = rho ** np.abs(idx[:, None] - idx[None, :])
    else:
        raise ValueError(f"Unknown covariance type: {covariance}")

    L = np.linalg.cholesky(Sigma)
    Z = rng.standard_normal(size=(n, d))
    return Z @ L.T


def gen_synthetic_qr_data(
    n,
    d,
    rng,
    tau,
    rho=0.0,
    covariance="toeplitz",
    sparsity=0.0,
    noise_dist="gaussian",
    sigma=1.0,
    heteroskedastic=False,
    hetero_strength=0.5,
    t_df=3,
):
    """
    Generate synthetic quantile regression data.

    Model:
        y = X beta_star + eps

    The synthetic experiments in the results notebook do not vary sparsity,
    but use the default sparsity=0.0.
    """
    X = sample_design_X(n, d, rng, covariance=covariance, rho=rho)

    beta_star = np.zeros(d)
    k = max(1, int(sparsity * d))
    support = rng.choice(d, size=k, replace=False)
    beta_star[support] = rng.normal(0.0, 1.0, size=k)

    mu = X @ beta_star
    base_sigma = float(sigma)

    if heteroskedastic:
        scale = base_sigma * (1.0 + float(hetero_strength) * np.abs(mu))
    else:
        scale = base_sigma

    if noise_dist == "gaussian":
        eps = rng.normal(0.0, scale, size=n)
        q_shift = scale * norm.ppf(tau)

    elif noise_dist == "laplace":
        b = scale / np.sqrt(2)
        eps = rng.laplace(0.0, b, size=n)
        q_shift = b * np.log(tau / (1.0 - tau))

    elif noise_dist == "student_t":
        df = float(t_df)
        eps = rng.standard_t(df, size=n) * scale
        q_shift = scale * t.ppf(tau, df)

    elif noise_dist == "logistic":
        eps = rng.logistic(0.0, scale, size=n)
        q_shift = logistic.ppf(tau, loc=0.0, scale=scale)

    elif noise_dist == "cauchy":
        eps = rng.standard_cauchy(size=n) * scale
        q_shift = cauchy.ppf(tau, loc=0.0, scale=scale)

    elif noise_dist == "gumbel":
        eps = rng.gumbel(0.0, scale, size=n)
        q_shift = gumbel_r.ppf(tau, loc=0.0, scale=scale)

    else:
        raise ValueError(f"Unknown noise_dist: {noise_dist}")

    y = mu + eps
    y_true_tau = mu + q_shift

    if heteroskedastic:
        intercept_true = None
    else:
        intercept_true = float(np.asarray(q_shift).reshape(-1)[0])

    return X, y, beta_star, intercept_true, y_true_tau, mu


def true_quantile_from_components(
    X,
    beta_star,
    tau,
    *,
    noise_dist="gaussian",
    sigma=1.0,
    heteroskedastic=False,
    hetero_strength=0.5,
    t_df=3,
):
    """
    Compute the pointwise true conditional tau-quantile from X and beta_star.
    """
    X = np.asarray(X)
    beta_star = np.asarray(beta_star).reshape(-1)

    mu = X @ beta_star
    base_sigma = float(sigma)

    if heteroskedastic:
        scale = base_sigma * (1.0 + float(hetero_strength) * np.abs(mu))
    else:
        scale = base_sigma

    if noise_dist == "gaussian":
        q_shift = scale * norm.ppf(tau)

    elif noise_dist == "laplace":
        b = scale / np.sqrt(2)
        q_shift = b * np.log(tau / (1.0 - tau))

    elif noise_dist == "student_t":
        q_shift = scale * t.ppf(tau, float(t_df))

    elif noise_dist == "logistic":
        q_shift = logistic.ppf(tau, loc=0.0, scale=scale)

    elif noise_dist == "cauchy":
        q_shift = cauchy.ppf(tau, loc=0.0, scale=scale)

    elif noise_dist == "gumbel":
        q_shift = gumbel_r.ppf(tau, loc=0.0, scale=scale)

    else:
        raise ValueError(f"Unknown noise_dist: {noise_dist}")

    return mu + q_shift

## Benchmark data

In [ ]:
# Benchmark loaders
from sklearn.datasets import fetch_california_housing

def load_engel():
    """
    Load Engel food expenditure dataset from statsmodels.

    Returns
    -------
    X : ndarray, shape (n, 1)
    y : ndarray, shape (n,)
    feature_names : list[str]
    """
    data = sm.datasets.engel.load_pandas().data
    X = data[["income"]].to_numpy()
    y = data["foodexp"].to_numpy()
    return X, y, ["income"]


def load_abalone():
    """
    Load Abalone dataset from UCI-style URL or local fallback if already downloaded.

    Returns
    -------
    X : ndarray
    y : ndarray
    feature_names : list[str]
    """
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"

    cols = [
        "Sex", "Length", "Diameter", "Height", "WholeWeight",
        "ShuckedWeight", "VisceraWeight", "ShellWeight", "Rings"
    ]

    df = pd.read_csv(url, names=cols)
    df = pd.get_dummies(df, columns=["Sex"], drop_first=True)

    y = df["Rings"].to_numpy()
    X = df.drop(columns=["Rings"]).to_numpy()
    feature_names = df.drop(columns=["Rings"]).columns.tolist()

    return X, y, feature_names


def load_concrete():
    """
    Load Concrete Compressive Strength dataset.

    Returns
    -------
    X : ndarray
    y : ndarray
    feature_names : list[str]
    """
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"

    df = pd.read_excel(url)

    y_col = df.columns[-1]
    y = df[y_col].to_numpy()
    X = df.drop(columns=[y_col]).to_numpy()
    feature_names = df.drop(columns=[y_col]).columns.tolist()

    return X, y, feature_names


def load_california_housing(remove_censored=True):
    """
    Load California Housing dataset.

    Parameters
    ----------
    remove_censored : bool
        If True, remove observations with target value equal to the censoring cap.

    Returns
    -------
    X : ndarray
    y : ndarray
    feature_names : list[str]
    """

    data = fetch_california_housing(as_frame=True)
    df = data.frame.copy()

    if remove_censored:
        df = df[df["MedHouseVal"] < 5.0].copy()

    y = df["MedHouseVal"].to_numpy()
    X = df.drop(columns=["MedHouseVal"]).to_numpy()
    feature_names = df.drop(columns=["MedHouseVal"]).columns.tolist()

    return X, y, feature_names


def load_dataset(name, **kwargs):
    """
    Unified dataset loader.

    Parameters
    ----------
    name : str
        One of {'engel', 'abalone', 'concrete', 'california'}.

    Returns
    -------
    X : ndarray
    y : ndarray
    feature_names : list[str]
    """
    name = name.lower()

    if name == "engel":
        return load_engel()

    if name == "abalone":
        return load_abalone()

    if name == "concrete":
        return load_concrete()

    if name in {"california", "california_housing", "cal_housing"}:
        return load_california_housing(**kwargs)

    raise ValueError(f"Unknown dataset name: {name}")

# Fit by method functions

## Standard results dictionary

In [ ]:
def _fit_result_dict(
    *,
    method,
    tau,
    model,
    coef,
    intercept,
    train_pred,
    test_pred,
    fit_seconds,
    n_iter=None,
    X_train=None,
    y_train=None,
    X_test=None,
    y_test=None,
):
    """
    Standardized result dictionary returned by all quantile-regression fitters.
    """
    result = {
        "method": method,
        "tau": tau,
        "model": model,
        "coef": coef,
        "beta": coef,  # alias for older CPS code
        "intercept": intercept,
        "train_pred": train_pred,
        "test_pred": test_pred,
        "yhat_train": train_pred,  # alias for older CPS code
        "yhat_test": test_pred,
        "fit_seconds": fit_seconds,
        "fit_time_sec": fit_seconds,
        "n_iter": n_iter,
    }

    if y_train is not None and train_pred is not None:
        result["train_pinball"] = pinball_loss(y_train, train_pred, tau)

    if y_test is not None and test_pred is not None:
        result["test_pinball"] = pinball_loss(y_test, test_pred, tau)

    if X_train is not None and y_train is not None:
        result["train_kkt_grad_inf"] = kkt_grad_inf(
            X_train, y_train, coef, intercept, tau
        )

    if X_test is not None and y_test is not None:
        result["test_kkt_grad_inf"] = kkt_grad_inf(
            X_test, y_test, coef, intercept, tau
        )

    return result

## Fit LP

In [ ]:
def fit_lp_quantile(
    X_train,
    y_train,
    X_test=None,
    y_test=None,
    tau=0.5,
    alpha=0.0,
    solver="highs",
    fit_intercept=True,
    solver_options=None,
):
    """
    Fit LP-based quantile regression using sklearn QuantileRegressor.
    """
    solver_options = {} if solver_options is None else solver_options

    model = QuantileRegressor(
        quantile=tau,
        alpha=alpha,
        solver=solver,
        fit_intercept=fit_intercept,
        solver_options=solver_options,
    )

    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - t0

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test) if X_test is not None else None

    return _fit_result_dict(
        method="lp",
        tau=tau,
        model=model,
        coef=np.asarray(model.coef_),
        intercept=float(model.intercept_),
        train_pred=train_pred,
        test_pred=test_pred,
        fit_seconds=fit_seconds,
        n_iter=getattr(model, "n_iter_", None),
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
    )

## Fit IRLS

In [ ]:
def fit_irls_quantile(
    X_train,
    y_train,
    X_test=None,
    y_test=None,
    tau=0.5,
    max_iter=3_000,
    p_tol=1e-6,
):
    """
    Fit IRLS quantile regression using statsmodels QuantReg.
    """
    X_train_sm = sm.add_constant(X_train, has_constant="add")
    X_test_sm = sm.add_constant(X_test, has_constant="add") if X_test is not None else None

    model = sm.QuantReg(y_train, X_train_sm)

    t0 = time.perf_counter()
    res = model.fit(q=tau, max_iter=max_iter, p_tol=p_tol)
    fit_seconds = time.perf_counter() - t0

    params = np.asarray(res.params)
    intercept = float(params[0])
    coef = params[1:]

    train_pred = X_train_sm @ params
    test_pred = X_test_sm @ params if X_test_sm is not None else None

    n_iter = getattr(res, "iterations", None)

    return _fit_result_dict(
        method="irls",
        tau=tau,
        model=res,
        coef=coef,
        intercept=intercept,
        train_pred=train_pred,
        test_pred=test_pred,
        fit_seconds=fit_seconds,
        n_iter=n_iter,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
    )

## Fit proxSGD

In [ ]:
def fit_proxsgd_quantile(
    X_train,
    y_train,
    X_test=None,
    y_test=None,
    tau=0.5,
    sgd_kwargs=None,
):
    """
    Fit proxSGD quantile regression using SGDQuantileRegressor.
    """
    sgd_kwargs = {} if sgd_kwargs is None else dict(sgd_kwargs)
    sgd_kwargs["quantile"] = tau

    model = SGDQuantileRegressor(**sgd_kwargs)

    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - t0

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test) if X_test is not None else None

    return _fit_result_dict(
        method="proxsgd",
        tau=tau,
        model=model,
        coef=np.asarray(model.coef_),
        intercept=float(model.intercept_),
        train_pred=train_pred,
        test_pred=test_pred,
        fit_seconds=fit_seconds,
        n_iter=getattr(model, "n_iter_", None),
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
    )

## Fit by method dispatcher

In [ ]:
def fit_quantile_method(
    method,
    X_train,
    y_train,
    X_test=None,
    y_test=None,
    tau=0.5,
    *,
    lp_alpha=0.0,
    lp_solver="highs",
    lp_solver_options=None,
    irls_max_iter=10_000,
    irls_p_tol=1e-6,
    sgd_kwargs=None,
):
    """
    Unified dispatcher for LP, IRLS, and proxSGD quantile-regression fits.
    """
    method_key = method.lower()

    if method_key in {"lp", "sklearn", "sklearn_lp"}:
        return fit_lp_quantile(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            tau=tau,
            alpha=lp_alpha,
            solver=lp_solver,
            solver_options=lp_solver_options,
        )

    if method_key in {"irls", "statsmodels", "quantreg"}:
        return fit_irls_quantile(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            tau=tau,
            max_iter=irls_max_iter,
            p_tol=irls_p_tol,
        )

    if method_key in {"proxsgd", "sgd", "prox_sgd"}:
        return fit_proxsgd_quantile(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            tau=tau,
            sgd_kwargs=sgd_kwargs,
        )

    raise ValueError(
        f"Unknown method '{method}'. Expected one of: 'lp', 'irls', 'proxsgd'."
    )

# Synthetic experiment runners

## Helpers

In [ ]:
def make_sgd_kwargs(
    *,
    tau,
    base_lr=0.5,
    batch_size=256,
    max_iter=3000,
    l1=0.0,
    l2=0.0,
    eval_every=500,
    use_adagrad=True,
    adagrad_eps=1e-8,
    use_averaging=True,
    dtype=np.float64,
    random_state=None,
    verbose=False,
    **extra_kwargs,
):
    """
    Build kwargs for SGDQuantileRegressor in one consistent place.
    """
    kwargs = dict(
        quantile=tau,
        max_iter=max_iter,
        base_lr=base_lr,
        batch_size=batch_size,
        l1=l1,
        l2=l2,
        eval_every=eval_every,
        use_adagrad=use_adagrad,
        adagrad_eps=adagrad_eps,
        use_averaging=use_averaging,
        dtype=dtype,
        random_state=random_state,
        verbose=verbose,
    )

    kwargs.update(extra_kwargs)
    return kwargs


def _get_row_value(row, name, default=None):
    """
    Helper for reading experiment-table rows with defaults.
    """
    return row[name] if name in row and pd.notna(row[name]) else default


def _method_list_from_row(row, default_methods=("lp", "irls", "proxsgd")):
    """
    Read methods from an experiment-table row, if provided.
    """
    methods = _get_row_value(row, "methods", default_methods)

    if isinstance(methods, str):
        methods = [m.strip() for m in methods.split(",")]

    return tuple(methods)

## Preprocess

In [ ]:
def split_and_standardize(X, y, test_size=1000, random_state=None):
    """
    Train/test split followed by standardization fit on the training data.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test, scaler

## Fit by quantile

In [ ]:
def run_synthetic_point_experiment(
    row,
    seed,
    methods=("lp", "irls", "proxsgd"),
    test_size=1000,
    default_sgd_kwargs=None,
):
    """
    Run one synthetic point-quantile experiment for one experiment-table row and seed.

    Returns one row per method.
    """
    default_sgd_kwargs = {} if default_sgd_kwargs is None else dict(default_sgd_kwargs)

    experiment = _get_row_value(row, "experiment", "synthetic")
    experiment_group = _get_row_value(row, "experiment_group", None)

    n = int(_get_row_value(row, "n", 10_000))
    d = int(_get_row_value(row, "d", 100))
    tau = float(_get_row_value(row, "tau", 0.5))

    rho = float(_get_row_value(row, "rho", 0.0))
    covariance = _get_row_value(row, "covariance", "toeplitz")
    noise_dist = _get_row_value(row, "noise_dist", "gaussian")
    sigma = float(_get_row_value(row, "sigma", 1.0))
    heteroskedastic = bool(_get_row_value(row, "heteroskedastic", False))
    hetero_strength = float(_get_row_value(row, "hetero_strength", 0.5))
    t_df = float(_get_row_value(row, "t_df", 3))

    sparsity = float(_get_row_value(row, "sparsity", 0.2))

    lp_alpha = float(_get_row_value(row, "lp_alpha", 0.0))
    l1 = float(_get_row_value(row, "l1", lp_alpha))
    l2 = float(_get_row_value(row, "l2", 0.0))

    base_lr = float(_get_row_value(row, "base_lr", default_sgd_kwargs.get("base_lr", 0.5)))
    batch_size = int(_get_row_value(row, "batch_size", default_sgd_kwargs.get("batch_size", 256)))
    max_iter = int(_get_row_value(row, "max_iter", default_sgd_kwargs.get("max_iter", 3000)))
    eval_every = int(_get_row_value(row, "eval_every", default_sgd_kwargs.get("eval_every", 500)))

    rng = np.random.default_rng(seed)

    X, y, beta_star, intercept_true, y_true_tau, mu = gen_synthetic_qr_data(
        n=n,
        d=d,
        rng=rng,
        tau=tau,
        rho=rho,
        covariance=covariance,
        sparsity=sparsity,
        noise_dist=noise_dist,
        sigma=sigma,
        heteroskedastic=heteroskedastic,
        hetero_strength=hetero_strength,
        t_df=t_df,
    )

    X_train, X_test, y_train, y_test, scaler = split_and_standardize(
        X,
        y,
        test_size=test_size,
        random_state=seed,
    )

    results = []

    for method in methods:
        method_key = method.lower()

        if method_key == "irls" and lp_alpha > 0:
            continue

        sgd_kwargs = make_sgd_kwargs(
            tau=tau,
            base_lr=base_lr,
            batch_size=batch_size,
            max_iter=max_iter,
            l1=l1,
            l2=l2,
            eval_every=eval_every,
            random_state=seed,
            **{
                k: v
                for k, v in default_sgd_kwargs.items()
                if k not in {
                    "quantile", "base_lr", "batch_size", "max_iter",
                    "l1", "l2", "eval_every", "random_state"
                }
            },
        )

        fit = fit_quantile_method(
            method=method_key,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            tau=tau,
            lp_alpha=lp_alpha,
            sgd_kwargs=sgd_kwargs,
        )

        results.append({
            "experiment": experiment,
            "experiment_group": experiment_group,
            "seed": seed,
            "method": fit["method"],
            "tau": tau,
            "n": n,
            "n_train": X_train.shape[0],
            "n_test": X_test.shape[0],
            "d": d,
            "rho": rho,
            "covariance": covariance,
            "noise_dist": noise_dist,
            "sigma": sigma,
            "heteroskedastic": heteroskedastic,
            "hetero_strength": hetero_strength,
            "t_df": t_df,
            "sparsity": sparsity,
            "lp_alpha": lp_alpha,
            "l1": l1,
            "l2": l2,
            "base_lr": base_lr,
            "batch_size": batch_size,
            "max_iter": max_iter,
            "fit_seconds": fit["fit_seconds"],
            "n_iter": fit["n_iter"],
            "train_pinball": fit.get("train_pinball", np.nan),
            "test_pinball": fit.get("test_pinball", np.nan),
            "train_kkt_grad_inf": fit.get("train_kkt_grad_inf", np.nan),
            "test_kkt_grad_inf": fit.get("test_kkt_grad_inf", np.nan),
        })

    return pd.DataFrame(results)

## Fit interval

In [ ]:
def run_synthetic_interval_experiment(
    row,
    seed,
    methods=("lp", "irls", "proxsgd"),
    test_size=1000,
    default_sgd_kwargs=None,
):
    """
    Run one synthetic interval experiment for one experiment-table row and seed.

    Fits lower and upper quantile models for each method, then computes
    interval metrics.
    """
    default_sgd_kwargs = {} if default_sgd_kwargs is None else dict(default_sgd_kwargs)

    experiment = _get_row_value(row, "experiment", "synthetic")
    experiment_group = _get_row_value(row, "experiment_group", None)

    n = int(_get_row_value(row, "n", 10_000))
    d = int(_get_row_value(row, "d", 100))

    lower_tau = float(_get_row_value(row, "lower_tau", 0.025))
    upper_tau = float(_get_row_value(row, "upper_tau", 0.975))
    alpha_interval = upper_tau - lower_tau
    miscoverage = 1.0 - alpha_interval

    rho = float(_get_row_value(row, "rho", 0.0))
    covariance = _get_row_value(row, "covariance", "toeplitz")
    noise_dist = _get_row_value(row, "noise_dist", "gaussian")
    sigma = float(_get_row_value(row, "sigma", 1.0))
    heteroskedastic = bool(_get_row_value(row, "heteroskedastic", False))
    hetero_strength = float(_get_row_value(row, "hetero_strength", 0.5))
    t_df = float(_get_row_value(row, "t_df", 3))

    sparsity = float(_get_row_value(row, "sparsity", 0.2))

    lp_alpha = float(_get_row_value(row, "lp_alpha", 0.0))
    l1 = float(_get_row_value(row, "l1", lp_alpha))
    l2 = float(_get_row_value(row, "l2", 0.0))

    base_lr = float(_get_row_value(row, "base_lr", default_sgd_kwargs.get("base_lr", 0.5)))
    batch_size = int(_get_row_value(row, "batch_size", default_sgd_kwargs.get("batch_size", 256)))
    max_iter = int(_get_row_value(row, "max_iter", default_sgd_kwargs.get("max_iter", 3000)))
    eval_every = int(_get_row_value(row, "eval_every", default_sgd_kwargs.get("eval_every", 500)))

    rng = np.random.default_rng(seed)

    X, y, beta_star, _, _, mu = gen_synthetic_qr_data(
        n=n,
        d=d,
        rng=rng,
        tau=0.5,
        rho=rho,
        covariance=covariance,
        sparsity=sparsity,
        noise_dist=noise_dist,
        sigma=sigma,
        heteroskedastic=heteroskedastic,
        hetero_strength=hetero_strength,
        t_df=t_df,
    )

    X_train, X_test, y_train, y_test, scaler = split_and_standardize(
        X,
        y,
        test_size=test_size,
        random_state=seed,
    )

    results = []

    for method in methods:
        method_key = method.lower()

        if method_key == "irls" and lp_alpha > 0:
            continue

        fits = {}

        for tau in (lower_tau, upper_tau):
            sgd_kwargs = make_sgd_kwargs(
                tau=tau,
                base_lr=base_lr,
                batch_size=batch_size,
                max_iter=max_iter,
                l1=l1,
                l2=l2,
                eval_every=eval_every,
                random_state=seed,
                **{
                    k: v
                    for k, v in default_sgd_kwargs.items()
                    if k not in {
                        "quantile", "base_lr", "batch_size", "max_iter",
                        "l1", "l2", "eval_every", "random_state"
                    }
                },
            )

            fits[tau] = fit_quantile_method(
                method=method_key,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                tau=tau,
                lp_alpha=lp_alpha,
                sgd_kwargs=sgd_kwargs,
            )

        lower_fit = fits[lower_tau]
        upper_fit = fits[upper_tau]

        y_lower = lower_fit["test_pred"]
        y_upper = upper_fit["test_pred"]

        interval = interval_metrics_with_winkler(
            y_test,
            y_lower,
            y_upper,
            alpha=miscoverage,
        )

        results.append({
            "experiment": experiment,
            "experiment_group": experiment_group,
            "seed": seed,
            "method": method_key,
            "lower_tau": lower_tau,
            "upper_tau": upper_tau,
            "nominal_coverage": alpha_interval,
            "n": n,
            "n_train": X_train.shape[0],
            "n_test": X_test.shape[0],
            "d": d,
            "rho": rho,
            "covariance": covariance,
            "noise_dist": noise_dist,
            "sigma": sigma,
            "heteroskedastic": heteroskedastic,
            "hetero_strength": hetero_strength,
            "t_df": t_df,
            "sparsity": sparsity,
            "lp_alpha": lp_alpha,
            "l1": l1,
            "l2": l2,
            "base_lr": base_lr,
            "batch_size": batch_size,
            "max_iter": max_iter,
            "fit_seconds_lower": lower_fit["fit_seconds"],
            "fit_seconds_upper": upper_fit["fit_seconds"],
            "fit_seconds": lower_fit["fit_seconds"] + upper_fit["fit_seconds"],
            "n_iter_lower": lower_fit["n_iter"],
            "n_iter_upper": upper_fit["n_iter"],
            "lower_test_pinball": lower_fit.get("test_pinball", np.nan),
            "upper_test_pinball": upper_fit.get("test_pinball", np.nan),
            "mean_test_pinball": np.nanmean([
                lower_fit.get("test_pinball", np.nan),
                upper_fit.get("test_pinball", np.nan),
            ]),
            "lower_train_kkt_grad_inf": lower_fit.get("train_kkt_grad_inf", np.nan),
            "upper_train_kkt_grad_inf": upper_fit.get("train_kkt_grad_inf", np.nan),
            "coverage": interval["coverage"],
            "mean_width": interval["mean_width"],
            "crossing_rate": interval["crossing_rate"],
            "winkler": interval["winkler"],
        })

    return pd.DataFrame(results)

## Experiment by table

In [ ]:
def run_synthetic_experiment_table(
    experiment_table,
    *,
    mode="interval",
    methods=("lp", "irls", "proxsgd"),
    seeds=range(5),
    test_size=1000,
    default_sgd_kwargs=None,
):
    """
    Run a full synthetic experiment table.

    Parameters
    ----------
    experiment_table : pandas.DataFrame
        One row per synthetic regime/configuration.
    mode : {'point', 'interval'}
        Whether to fit one tau per row or lower/upper quantiles per row.
    methods : tuple[str]
        Methods to evaluate. Options: 'lp', 'irls', 'proxsgd'.
    seeds : iterable
        Random seeds.
    test_size : int or float
        Test-set size passed to train_test_split.
    default_sgd_kwargs : dict or None
        Default proxSGD keyword arguments.

    Returns
    -------
    results : pandas.DataFrame
        Tidy synthetic experiment results.
    """
    all_results = []

    for _, row in experiment_table.iterrows():
        row_methods = _method_list_from_row(row, default_methods=methods)

        for seed in seeds:
            if mode == "point":
                out = run_synthetic_point_experiment(
                    row=row,
                    seed=seed,
                    methods=row_methods,
                    test_size=test_size,
                    default_sgd_kwargs=default_sgd_kwargs,
                )

            elif mode == "interval":
                out = run_synthetic_interval_experiment(
                    row=row,
                    seed=seed,
                    methods=row_methods,
                    test_size=test_size,
                    default_sgd_kwargs=default_sgd_kwargs,
                )

            else:
                raise ValueError("mode must be either 'point' or 'interval'.")

            all_results.append(out)

    return pd.concat(all_results, ignore_index=True)

# Benchmark experiment runners

## Preprocess

In [ ]:
def prepare_benchmark_split(
    dataset,
    *,
    test_size=0.2,
    random_state=33,
    scale=True,
    loader_kwargs=None,
):
    """
    Load a benchmark dataset, create a train/test split, and optionally standardize X.

    Returns
    -------
    dict
        Contains X_train, X_test, y_train, y_test, scaler, feature_names.
    """
    loader_kwargs = {} if loader_kwargs is None else dict(loader_kwargs)

    X, y, feature_names = load_dataset(dataset, **loader_kwargs)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
    )

    scaler = None
    if scale:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    return {
        "dataset": dataset,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "scaler": scaler,
        "feature_names": feature_names,
    }

## Fit by quantile

In [ ]:
def run_benchmark_quantile_fits(
    dataset,
    *,
    taus=(0.1, 0.5, 0.9),
    methods=("lp", "irls", "proxsgd"),
    test_size=0.2,
    random_state=33,
    scale=True,
    loader_kwargs=None,
    lp_alpha=0.0,
    default_sgd_kwargs=None,
    irls_max_iter=10_000,
    irls_p_tol=1e-6,
):
    """
    Fit each method at each tau for one benchmark dataset.

    Returns
    -------
    fit_results : dict
        Nested dictionary fit_results[method][tau] = fit result dict.
    point_rows : pandas.DataFrame
        One row per dataset/method/tau.
    split : dict
        Train/test split and metadata.
    """
    default_sgd_kwargs = {} if default_sgd_kwargs is None else dict(default_sgd_kwargs)

    split = prepare_benchmark_split(
        dataset,
        test_size=test_size,
        random_state=random_state,
        scale=scale,
        loader_kwargs=loader_kwargs,
    )

    X_train = split["X_train"]
    X_test = split["X_test"]
    y_train = split["y_train"]
    y_test = split["y_test"]

    fit_results = {}
    rows = []

    for method in methods:
        method_key = method.lower()

        if method_key == "irls" and lp_alpha > 0:
            continue

        fit_results[method_key] = {}

        for tau in taus:
            tau = float(tau)

            sgd_kwargs = dict(default_sgd_kwargs)
            sgd_kwargs["quantile"] = tau
            sgd_kwargs.setdefault("random_state", random_state)

            fit = fit_quantile_method(
                method=method_key,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                tau=tau,
                lp_alpha=lp_alpha,
                irls_max_iter=irls_max_iter,
                irls_p_tol=irls_p_tol,
                sgd_kwargs=sgd_kwargs,
            )

            fit_results[method_key][tau] = fit

            rows.append({
                "dataset": dataset,
                "random_state": random_state,
                "method": fit["method"],
                "tau": tau,
                "n_train": X_train.shape[0],
                "n_test": X_test.shape[0],
                "d": X_train.shape[1],
                "lp_alpha": lp_alpha,
                "l1": sgd_kwargs.get("l1", lp_alpha),
                "base_lr": sgd_kwargs.get("base_lr", np.nan),
                "batch_size": sgd_kwargs.get("batch_size", np.nan),
                "max_iter": sgd_kwargs.get("max_iter", np.nan),
                "fit_seconds": fit["fit_seconds"],
                "n_iter": fit["n_iter"],
                "train_pinball": fit.get("train_pinball", np.nan),
                "test_pinball": fit.get("test_pinball", np.nan),
                "train_kkt_grad_inf": fit.get("train_kkt_grad_inf", np.nan),
                "test_kkt_grad_inf": fit.get("test_kkt_grad_inf", np.nan),
                "rmse": rmse(y_test, fit["test_pred"]),
            })

    return fit_results, pd.DataFrame(rows), split

## Interval

In [ ]:
def run_benchmark_interval_experiment(
    dataset,
    *,
    lower_tau=0.1,
    upper_tau=0.9,
    methods=("lp", "irls", "proxsgd"),
    test_size=0.2,
    random_state=33,
    scale=True,
    loader_kwargs=None,
    lp_alpha=0.0,
    default_sgd_kwargs=None,
    irls_max_iter=10_000,
    irls_p_tol=1e-6,
):
    """
    Run benchmark interval experiment for one dataset.

    Fits lower, median, and upper quantiles when available, and reports:
    pinball loss, RMSE at tau=0.5, coverage, width, crossing, and Winkler.
    """
    taus = sorted(set([float(lower_tau), 0.5, float(upper_tau)]))

    fit_results, point_df, split = run_benchmark_quantile_fits(
        dataset,
        taus=taus,
        methods=methods,
        test_size=test_size,
        random_state=random_state,
        scale=scale,
        loader_kwargs=loader_kwargs,
        lp_alpha=lp_alpha,
        default_sgd_kwargs=default_sgd_kwargs,
        irls_max_iter=irls_max_iter,
        irls_p_tol=irls_p_tol,
    )

    y_test = split["y_test"]
    miscoverage = 1.0 - (upper_tau - lower_tau)

    rows = []

    for method_key, fits_by_tau in fit_results.items():
        if lower_tau not in fits_by_tau or upper_tau not in fits_by_tau:
            continue

        lower_fit = fits_by_tau[float(lower_tau)]
        upper_fit = fits_by_tau[float(upper_tau)]
        median_fit = fits_by_tau.get(0.5, None)

        y_lower = lower_fit["test_pred"]
        y_upper = upper_fit["test_pred"]

        interval = interval_metrics_with_winkler(
            y_test,
            y_lower,
            y_upper,
            alpha=miscoverage,
        )

        rows.append({
            "dataset": dataset,
            "random_state": random_state,
            "method": method_key,
            "lower_tau": float(lower_tau),
            "upper_tau": float(upper_tau),
            "nominal_coverage": float(upper_tau - lower_tau),
            "n_train": split["X_train"].shape[0],
            "n_test": split["X_test"].shape[0],
            "d": split["X_train"].shape[1],
            "lp_alpha": lp_alpha,
            "l1": lower_fit.get("model").l1 if hasattr(lower_fit.get("model"), "l1") else lp_alpha,
            "fit_seconds": lower_fit["fit_seconds"] + upper_fit["fit_seconds"],
            "fit_seconds_lower": lower_fit["fit_seconds"],
            "fit_seconds_upper": upper_fit["fit_seconds"],
            "lower_test_pinball": lower_fit.get("test_pinball", np.nan),
            "upper_test_pinball": upper_fit.get("test_pinball", np.nan),
            "mean_test_pinball": np.nanmean([
                lower_fit.get("test_pinball", np.nan),
                upper_fit.get("test_pinball", np.nan),
            ]),
            "rmse": rmse(y_test, median_fit["test_pred"]) if median_fit is not None else np.nan,
            "coverage": interval["coverage"],
            "mean_width": interval["mean_width"],
            "crossing_rate": interval["crossing_rate"],
            "winkler": interval["winkler"],
            "lower_train_kkt_grad_inf": lower_fit.get("train_kkt_grad_inf", np.nan),
            "upper_train_kkt_grad_inf": upper_fit.get("train_kkt_grad_inf", np.nan),
        })

    return pd.DataFrame(rows), point_df, fit_results, split

## Experiment by table

In [ ]:
def run_benchmark_experiment_table(
    experiment_table,
    *,
    methods=("lp", "irls", "proxsgd"),
    test_size=0.2,
    default_sgd_kwargs=None,
    scale=True,
):
    """
    Run benchmark interval experiments from an experiment table.

    Expected useful columns include:
        dataset, lower_tau, upper_tau, random_state, lp_alpha, l1,
        test_size, methods
    """
    all_interval = []
    all_point = []

    for _, row in experiment_table.iterrows():
        dataset = _get_row_value(row, "dataset")
        if dataset is None:
            raise ValueError("Each benchmark experiment row must include a 'dataset' column.")

        row_methods = _method_list_from_row(row, default_methods=methods)

        lower_tau = float(_get_row_value(row, "lower_tau", 0.1))
        upper_tau = float(_get_row_value(row, "upper_tau", 0.9))
        random_state = int(_get_row_value(row, "random_state", 123))
        row_test_size = _get_row_value(row, "test_size", test_size)
        lp_alpha = float(_get_row_value(row, "lp_alpha", 0.0))
        l1 = float(_get_row_value(row, "l1", lp_alpha))

        row_sgd_kwargs = {} if default_sgd_kwargs is None else dict(default_sgd_kwargs)
        row_sgd_kwargs["l1"] = l1

        for key in ["base_lr", "batch_size", "max_iter", "eval_every", "l2"]:
            if key in row and pd.notna(row[key]):
                row_sgd_kwargs[key] = row[key]

        interval_df, point_df, _, _ = run_benchmark_interval_experiment(
            dataset=dataset,
            lower_tau=lower_tau,
            upper_tau=upper_tau,
            methods=row_methods,
            test_size=row_test_size,
            random_state=random_state,
            scale=scale,
            lp_alpha=lp_alpha,
            default_sgd_kwargs=row_sgd_kwargs,
        )

        for col in ["experiment", "experiment_group"]:
            if col in row:
                interval_df[col] = row[col]
                point_df[col] = row[col]

        all_interval.append(interval_df)
        all_point.append(point_df)

    return {
        "interval": pd.concat(all_interval, ignore_index=True),
        "point": pd.concat(all_point, ignore_index=True),
    }

# CPS experiment runners

## Preprocess

In [ ]:
def prepare_cps_features(df):
    """
    Construct CPS feature matrix exactly as used in the paper.

    Model:
        log_wage ~ AGE + AGE^2 + EDUC
                   + SEX + RACE + HISPAN + REGION
                   + STATE + OCC + IND
    """
    df = df.copy()

    y = df["log_wage"].to_numpy()

    df["AGE2"] = df["AGE"] ** 2

    X_cont = df[["AGE", "AGE2", "EDUC"]]

    X_cat = pd.get_dummies(
        df[
            [
                "SEX",
                "RACE",
                "HISPAN",
                "REGION",
                "STATE",
                "OCC",
                "IND",
            ]
        ],
        drop_first=True,
    )

    X = pd.concat([X_cont, X_cat], axis=1)

    return X.to_numpy(), y, X.columns.tolist()

In [ ]:
def make_cps_train_test_split(
    df,
    *,
    test_size=10_000,
    random_state=123,
):
    """
    Create CPS train/test split with fixed test size.
    """
    X, y, feature_names = prepare_cps_features(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    return {
        "X_train_full": X_train,
        "y_train_full": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "scaler": scaler,
        "feature_names": feature_names,
    }

## Fit by quantile

In [ ]:
def run_cps_quantile_fits(
    X_train,
    y_train,
    X_test,
    y_test,
    *,
    taus=(0.1, 0.5, 0.9),
    methods=("lp", "irls", "proxsgd"),
    lp_alpha=0.0,
    lp_solver="highs",
    lp_solver_options=None,
    irls_max_iter=3_000,
    irls_p_tol=1e-6,
    sgd_kwargs=None,
    seed=123,
):
    sgd_kwargs = {} if sgd_kwargs is None else dict(sgd_kwargs)

    fit_results = {}

    for method in methods:
        method_key = method.lower()

        if method_key == "irls" and lp_alpha > 0:
            continue

        fit_results[method_key] = {}

        for tau in taus:
            tau = float(tau)

            kwargs = dict(sgd_kwargs)
            kwargs["quantile"] = tau
            kwargs.setdefault("random_state", seed)

            fit = fit_quantile_method(
                method=method_key,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                tau=tau,
                lp_alpha=lp_alpha,
                lp_solver=lp_solver,
                lp_solver_options=lp_solver_options,
                irls_max_iter=irls_max_iter,
                irls_p_tol=irls_p_tol,
                sgd_kwargs=kwargs,
            )

            fit_results[method_key][tau] = fit

    return fit_results

## Scaling experiment

In [ ]:
def run_cps_scaling_experiment(
    df,
    *,
    N_SUB_list,
    taus=(0.1, 0.5, 0.9),
    methods=("lp", "irls", "proxsgd"),
    seed=123,
    test_size=10_000,
    lp_alpha=0.0,
    lp_solver="highs",
    lp_solver_options=None,
    irls_max_iter=3_000,
    irls_p_tol=1e-6,
    sgd_kwargs=None,
):
    base = make_cps_train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
    )

    X_full = base["X_train_full"]
    y_full = base["y_train_full"]
    X_test = base["X_test"]
    y_test = base["y_test"]

    results = []

    for N_SUB in N_SUB_list:
        X_train = X_full[:N_SUB]
        y_train = y_full[:N_SUB]

        fits = run_cps_quantile_fits(
            X_train,
            y_train,
            X_test,
            y_test,
            taus=taus,
            methods=methods,
            lp_alpha=lp_alpha,
            lp_solver=lp_solver,
            lp_solver_options=lp_solver_options,
            irls_max_iter=irls_max_iter,
            irls_p_tol=irls_p_tol,
            sgd_kwargs=sgd_kwargs,
            seed=seed,
        )

        for method, tau_dict in fits.items():
            lower = tau_dict.get(0.1)
            median = tau_dict.get(0.5)
            upper = tau_dict.get(0.9)

            if lower is None or upper is None:
                continue

            y_lower = lower["test_pred"]
            y_upper = upper["test_pred"]

            interval = interval_metrics_with_winkler(
                y_test,
                y_lower,
                y_upper,
                alpha=0.2,
            )

            results.append({
                "method": method,
                "N_SUB": N_SUB,
                "seed": seed,
                "tau": 0.5,
                "n_train": N_SUB,
                "n_test": len(y_test),
                "d": X_train.shape[1],
                "fit_seconds": sum(
                    tau_dict[t]["fit_seconds"] for t in tau_dict
                ),
                "coverage": interval["coverage"],
                "mean_width": interval["mean_width"],
                "winkler": interval["winkler"],
                "rmse": rmse(y_test, median["test_pred"]) if median else np.nan,
            })

    return pd.DataFrame(results)

# Summary functions

## Helpers

In [ ]:
def format_mean_sd_pm(mean, sd, digits=2):
    if pd.isna(mean):
        return ""
    if pd.isna(sd):
        return f"{mean:.{digits}f}"

    return rf"{mean:.{digits}f} $\pm$ {sd:.{digits}f}"


def summarize_by_mean_sd(
    df,
    group_cols,
    metric_cols,
    digits=4,
):
    """
    Summarize selected metrics by mean and standard deviation.

    Returns one row per group, with entries formatted as mean ± sd.
    """
    summary = (
        df.groupby(group_cols, dropna=False)[metric_cols]
        .agg(["mean", "std"])
        .reset_index()
    )

    # Flatten multi-index columns
    summary.columns = [
        "_".join(col).strip("_") if isinstance(col, tuple) else col
        for col in summary.columns
    ]

    out = summary[group_cols].copy()

    for metric in metric_cols:
        out[metric] = [
            format_mean_sd_pm(m, s, digits=digits)
            for m, s in zip(summary[f"{metric}_mean"], summary[f"{metric}_std"])
        ]

    return out

def dataframe_to_latex_table(
    df,
    *,
    caption=None,
    label=None,
    index=False,
    column_format=None,
    float_format=None,
    escape=False,
):
    """
    Convert a summary DataFrame to LaTeX.

    This is intentionally thin so that paper-specific formatting can remain
    in the results notebook.
    """
    return df.to_latex(
        index=index,
        caption=caption,
        label=label,
        column_format=column_format,
        float_format=float_format,
        escape=escape,
    )

## Synthetic data

In [ ]:
def summarize_synthetic_intervals(
    df,
    *,
    group_cols=("experiment", "method"),
    metric_cols=(
        "mean_test_pinball",
        "coverage",
        "mean_width",
        "crossing_rate",
        "fit_seconds",
    ),
    digits=4,
):
    """
    Summarize synthetic interval experiments using mean ± one standard deviation.

    Parameters
    ----------
    df : pandas.DataFrame
        Output from run_synthetic_experiment_table(..., mode="interval").
    group_cols : tuple[str]
        Columns to group by, typically ("experiment", "method").
    metric_cols : tuple[str]
        Numeric metric columns to summarize.
    digits : int
        Number of decimal places.

    Returns
    -------
    pandas.DataFrame
        Summary table with formatted mean ± sd entries.
    """
    return summarize_by_mean_sd(
        df,
        group_cols=list(group_cols),
        metric_cols=list(metric_cols),
        digits=digits,
    )

## Benchmark

In [ ]:
def summarize_benchmark_intervals(
    df,
    *,
    group_cols=("dataset", "method"),
    metric_cols=(
        "mean_test_pinball",
        "rmse",
        "coverage",
        "mean_width",
        "fit_seconds",
    ),
    digits=4,
):
    """
    Summarize benchmark interval experiments.
    """
    group_cols = list(group_cols)
    metric_cols = list(metric_cols)

    return summarize_by_mean_sd(
        df,
        group_cols=group_cols,
        metric_cols=metric_cols,
        digits=digits,
    )


def summarize_benchmark_point(
    df,
    *,
    group_cols=("dataset", "method", "tau"),
    metric_cols=(
        "test_pinball",
        "rmse",
        "train_kkt_grad_inf",
        "fit_seconds",
    ),
    digits=4,
):
    """
    Summarize benchmark point-quantile fits.
    """
    group_cols = list(group_cols)
    metric_cols = list(metric_cols)

    return summarize_by_mean_sd(
        df,
        group_cols=group_cols,
        metric_cols=metric_cols,
        digits=digits,
    )

In [ ]:
def summarize_convergence_results(curves, refs):
    """
    Summarize final proxSGD losses and LP/IRLS reference losses.
    """
    final_curves = (
        curves
        .sort_values(["dataset", "seed", "split", "iteration"])
        .groupby(["dataset", "seed", "split"], as_index=False)
        .tail(1)
    )

    prox_summary = (
        final_curves
        .groupby(["dataset", "split"])["pinball_loss"]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
        .rename(
            columns={
                "mean": "proxSGD_final_mean",
                "std": "proxSGD_final_std",
                "min": "proxSGD_final_min",
                "max": "proxSGD_final_max",
            }
        )
    )

    ref_summary = (
        refs
        .groupby(["dataset", "method", "split"])["pinball_loss"]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
        .rename(
            columns={
                "mean": "ref_mean",
                "std": "ref_std",
                "min": "ref_min",
                "max": "ref_max",
            }
        )
    )

    return prox_summary, ref_summary


def compare_lp_irls_refs(refs):
    """
    Compare LP and IRLS reference losses seed-by-seed.
    """
    wide = (
        refs
        .pivot_table(
            index=["dataset", "seed", "split"],
            columns="method",
            values="pinball_loss",
        )
        .reset_index()
    )

    wide["abs_diff_LP_IRLS"] = np.abs(wide["LP"] - wide["IRLS"])

    summary = (
        wide
        .groupby(["dataset", "split"])["abs_diff_LP_IRLS"]
        .agg(["mean", "std", "max"])
        .reset_index()
        .rename(
            columns={
                "mean": "mean_abs_diff",
                "std": "std_abs_diff",
                "max": "max_abs_diff",
            }
        )
    )

    return summary

## CPS

In [ ]:
def summarize_cps_scaling(
    df,
    *,
    group_cols=("N_SUB", "method"),
    metric_cols=(
        "fit_seconds",
        "rmse",
        "coverage",
        "mean_width",
        "winkler",
    ),
    digits=4,
):
    """
    Summarize CPS scaling results.
    """
    group_cols = list(group_cols)
    metric_cols = list(metric_cols)

    return summarize_by_mean_sd(
        df,
        group_cols=group_cols,
        metric_cols=metric_cols,
        digits=digits,
    )

# Plotting

## Synthetic Plots

In [ ]:
def plot_metric_vs_lr_by_batch(
    df,
    *,
    metric_col="coverage",
    method_col="method",
    reference_method="lp",
    target_method="proxsgd",
    group_cols=("experiment", "seed"),
    lr_col="base_lr",
    bs_col="batch_size",
    agg="median",
    absolute=True,
    logx=True,
    figsize=(7, 5),
    ylabel=None,
    title=None,
):
    """
    Plot target-vs-reference metric gap against base learning rate,
    with one line per mini-batch size.

    Designed for synthetic robustness outputs from run_synthetic_experiment_table.

    Default:
        |proxSGD coverage - LP coverage|
    """
    required = [
        metric_col,
        method_col,
        lr_col,
        bs_col,
        *group_cols,
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(
            f"Missing columns {missing}. "
            f"Available columns include: {list(df.columns)[:25]} ..."
        )

    data = df.copy()
    data[method_col] = data[method_col].astype(str).str.lower()

    reference_method = reference_method.lower()
    target_method = target_method.lower()

    ref = (
        data[data[method_col] == reference_method]
        .loc[:, list(group_cols) + [metric_col]]
        .drop_duplicates(subset=list(group_cols))
        .rename(columns={metric_col: "reference_metric"})
    )

    target = data[data[method_col] == target_method].copy()

    if ref.empty:
        raise ValueError(f"No rows found for reference_method='{reference_method}'.")

    if target.empty:
        raise ValueError(f"No rows found for target_method='{target_method}'.")

    merged = target.merge(ref, on=list(group_cols), how="left")

    if merged["reference_metric"].isna().all():
        raise ValueError(
            "No matching reference rows found. "
            "Try adjusting group_cols, e.g. group_cols=('experiment', 'seed')."
        )

    merged["metric_gap"] = merged[metric_col] - merged["reference_metric"]

    if absolute:
        merged["metric_gap"] = merged["metric_gap"].abs()

    agg_fn = np.nanmedian if agg == "median" else np.nanmean

    grouped = (
        merged.groupby([bs_col, lr_col], dropna=False)["metric_gap"]
        .agg(agg_fn)
        .reset_index()
    )

    batch_sizes = sorted(grouped[bs_col].dropna().unique())

    plt.figure(figsize=figsize)

    for bs in batch_sizes:
        sub = grouped[grouped[bs_col] == bs].sort_values(lr_col)

        plt.plot(
            sub[lr_col],
            sub["metric_gap"],
            marker="o",
            linewidth=2,
            label=f"batch={int(bs)}",
        )

    if logx:
        plt.xscale("log")

    plt.xlabel(r"Base learning rate $\eta_0$", fontsize=14)

    if ylabel is None:
        ylabel = f"{'Absolute ' if absolute else ''}{metric_col} gap"

    plt.ylabel(ylabel, fontsize=16)

    if title is not None:
        plt.title(title, fontsize=16)

    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

def plot_optimizer_lossgap(
    df,
    *,
    metric_col="mean_test_pinball",
    method_col="method",
    reference_method="lp",
    target_method="proxsgd",
    group_cols=("experiment", "seed"),
    lr_col="base_lr",
    bs_col="batch_size",
    agg="median",
    figsize=(6, 5),
    cmap="viridis",
    percentile_clip=95,
):
    """
    Heatmap of absolute loss gap for target_method vs reference_method.

    Designed for outputs from run_synthetic_experiment_table(..., mode="interval").

    Computes:
        |target metric - reference metric|

    Default:
        |proxSGD mean_test_pinball - LP mean_test_pinball|
    """
    required = [
        metric_col,
        method_col,
        lr_col,
        bs_col,
        *group_cols,
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(
            f"Missing columns {missing}. "
            f"Available columns include: {list(df.columns)[:25]} ..."
        )

    data = df.copy()
    data[method_col] = data[method_col].str.lower()

    reference_method = reference_method.lower()
    target_method = target_method.lower()

    ref = (
        data[data[method_col] == reference_method]
        .loc[:, list(group_cols) + [metric_col]]
        .rename(columns={metric_col: "reference_metric"})
    )

    target = data[data[method_col] == target_method].copy()

    if ref.empty:
        raise ValueError(f"No rows found for reference_method='{reference_method}'.")

    if target.empty:
        raise ValueError(f"No rows found for target_method='{target_method}'.")

    merged = target.merge(ref, on=list(group_cols), how="left")

    if merged["reference_metric"].isna().all():
        raise ValueError(
            "No matching reference rows found. "
            "Try adjusting group_cols, e.g. group_cols=('experiment', 'seed')."
        )

    merged["abs_loss_gap"] = (
        merged[metric_col] - merged["reference_metric"]
    ).abs()

    agg_fn = np.nanmedian if agg == "median" else np.nanmean

    grouped = (
        merged.groupby([bs_col, lr_col], dropna=False)["abs_loss_gap"]
        .agg(agg_fn)
        .reset_index()
    )

    batch_sizes = np.sort(grouped[bs_col].dropna().unique())
    lrs = np.sort(grouped[lr_col].dropna().unique())

    loss_mat = (
        grouped.pivot(index=bs_col, columns=lr_col, values="abs_loss_gap")
        .loc[batch_sizes, lrs]
        .values
    )

    loss_vmin = np.nanmin(loss_mat)
    loss_vmax = np.nanpercentile(loss_mat, percentile_clip)

    plt.figure(figsize=figsize)

    im = plt.imshow(
        loss_mat,
        aspect="auto",
        origin="lower",
        cmap=cmap,
        vmin=loss_vmin,
        vmax=loss_vmax,
    )

    plt.xlabel(r"Base learning rate $\eta_0$", fontsize=16)
    plt.ylabel(r"Batch size $m$", fontsize=16)

    plt.xticks(range(len(lrs)))
    xticklabels = []
    for lr in lrs:
        exp = np.log10(lr)
        if lr > 0 and np.isclose(exp, round(exp)):
            xticklabels.append(rf"$10^{{{int(round(exp))}}}$")
        else:
            xticklabels.append(f"{lr:g}")
    plt.gca().set_xticklabels(xticklabels)

    plt.yticks(range(len(batch_sizes)), batch_sizes)

    cbar = plt.colorbar(im)
    cbar.set_label("Absolute loss gap", fontsize=16)

    plt.tight_layout()

## Convergence plot

In [ ]:
import matplotlib.ticker as mticker

In [ ]:
def collect_convergence_results_with_references(
    dataset_names,
    seeds,
    tau,
    estimator_cls,
    sgd_init_kwargs,
    sgd_fit_kwargs=None,
    lp_kwargs=None,
    irls_fit_kwargs=None,
    test_size=0.2,
    scale=True,
):
    sgd_fit_kwargs = {} if sgd_fit_kwargs is None else dict(sgd_fit_kwargs)
    lp_kwargs = {} if lp_kwargs is None else dict(lp_kwargs)
    irls_fit_kwargs = {} if irls_fit_kwargs is None else dict(irls_fit_kwargs)

    curve_rows = []
    ref_rows = []

    for dataset_name in dataset_names:
        for seed in seeds:
            split = prepare_benchmark_split(
                dataset_name,
                test_size=test_size,
                random_state=seed,
                scale=scale,
            )

            X_train_fit = split["X_train"]
            X_test_fit = split["X_test"]
            y_train = split["y_train"]
            y_test = split["y_test"]

            # -------------------------
            # LP reference
            # -------------------------
            lp_model = QuantileRegressor(
                quantile=tau,
                **lp_kwargs,
            )

            t0 = time.perf_counter()
            lp_model.fit(X_train_fit, y_train)
            lp_time = time.perf_counter() - t0

            lp_train_pred = lp_model.predict(X_train_fit)
            lp_test_pred = lp_model.predict(X_test_fit)

            ref_rows.extend(
                [
                    {
                        "dataset": dataset_name,
                        "seed": seed,
                        "method": "LP",
                        "split": "train",
                        "pinball_loss": pinball_loss(y_train, lp_train_pred, tau),
                        "fit_time_sec": lp_time,
                    },
                    {
                        "dataset": dataset_name,
                        "seed": seed,
                        "method": "LP",
                        "split": "test",
                        "pinball_loss": pinball_loss(y_test, lp_test_pred, tau),
                        "fit_time_sec": lp_time,
                    },
                ]
            )

            # -------------------------
            # IRLS reference
            # -------------------------
            X_train_sm = sm.add_constant(X_train_fit, has_constant="add")
            X_test_sm = sm.add_constant(X_test_fit, has_constant="add")

            irls_model = sm.QuantReg(y_train, X_train_sm)

            t0 = time.perf_counter()
            irls_res = irls_model.fit(q=tau, **irls_fit_kwargs)
            irls_time = time.perf_counter() - t0

            irls_train_pred = np.asarray(
                irls_res.predict(X_train_sm),
                dtype=float,
            )
            irls_test_pred = np.asarray(
                irls_res.predict(X_test_sm),
                dtype=float,
            )

            ref_rows.extend(
                [
                    {
                        "dataset": dataset_name,
                        "seed": seed,
                        "method": "IRLS",
                        "split": "train",
                        "pinball_loss": pinball_loss(y_train, irls_train_pred, tau),
                        "fit_time_sec": irls_time,
                    },
                    {
                        "dataset": dataset_name,
                        "seed": seed,
                        "method": "IRLS",
                        "split": "test",
                        "pinball_loss": pinball_loss(y_test, irls_test_pred, tau),
                        "fit_time_sec": irls_time,
                    },
                ]
            )

            # -------------------------
            # proxSGD convergence curve
            # -------------------------
            sgd_kwargs_seeded = dict(sgd_init_kwargs)
            sgd_kwargs_seeded["random_state"] = seed

            prox_model = estimator_cls(
                quantile=tau,
                **sgd_kwargs_seeded,
            )

            prox_model.fit(
                X_train_fit,
                y_train,
                X_monitor=X_test_fit,
                y_monitor=y_test,
                monitor_name="test",
                **sgd_fit_kwargs,
            )

            hist = prox_model.history_

            for it, train_loss, test_loss in zip(
                hist["iter"],
                hist["train_pinball_loss"],
                hist["monitor_pinball_loss"],
            ):
                curve_rows.append(
                    {
                        "dataset": dataset_name,
                        "seed": seed,
                        "method": "proxSGD",
                        "iteration": it,
                        "split": "train",
                        "pinball_loss": train_loss,
                    }
                )

                curve_rows.append(
                    {
                        "dataset": dataset_name,
                        "seed": seed,
                        "method": "proxSGD",
                        "iteration": it,
                        "split": "test",
                        "pinball_loss": test_loss,
                    }
                )

    curves = pd.DataFrame(curve_rows)
    references = pd.DataFrame(ref_rows)

    return curves, references

In [ ]:
def plot_convergence_with_references(
    curves,
    refs,
    dataset_names,
    dataset_labels=None,
    figsize_per_panel=(6, 5),
    use_log_scale=True,
):
    """
    Plot proxSGD train/test convergence curves with LP reference losses.

    Shaded bands show mean +/- one standard deviation across seeds.
    Horizontal black lines show mean LP train/test losses across seeds.
    """
    if dataset_labels is None:
        dataset_labels = {
            name: name.replace("_", " ").title()
            for name in dataset_names
        }

    fig, axes = plt.subplots(
        1,
        len(dataset_names),
        figsize=(figsize_per_panel[0] * len(dataset_names), figsize_per_panel[1]),
        sharex=False,
        sharey=False,
    )

    if len(dataset_names) == 1:
        axes = [axes]

    for ax, dataset in zip(axes, dataset_names):

        # -------------------------
        # proxSGD curves
        # -------------------------
        df = curves[
            (curves["dataset"] == dataset)
            & (curves["method"] == "proxSGD")
        ]

        train_df = (
            df[df["split"] == "train"]
            .pivot(index="seed", columns="iteration", values="pinball_loss")
            .sort_index(axis=1)
        )

        test_df = (
            df[df["split"] == "test"]
            .pivot(index="seed", columns="iteration", values="pinball_loss")
            .sort_index(axis=1)
        )

        iters = train_df.columns.to_numpy()

        train_mean = train_df.mean(axis=0).to_numpy()
        train_std = train_df.std(axis=0).to_numpy()

        test_mean = test_df.mean(axis=0).to_numpy()
        test_std = test_df.std(axis=0).to_numpy()

        ax.plot(
            iters,
            train_mean,
            label="proxSGD train",
            linewidth=2,
        )

        ax.plot(
            iters,
            test_mean,
            linestyle="--",
            label="proxSGD test",
            linewidth=2,
        )

        ax.fill_between(
            iters,
            train_mean - train_std,
            train_mean + train_std,
            alpha=0.2,
        )

        ax.fill_between(
            iters,
            test_mean - test_std,
            test_mean + test_std,
            alpha=0.2,
        )

        # -------------------------
        # LP references
        # -------------------------
        lp_df = refs[
            (refs["dataset"] == dataset)
            & (refs["method"] == "LP")
        ]

        lp_train_mean = (
            lp_df[lp_df["split"] == "train"]["pinball_loss"]
            .mean()
        )

        lp_test_mean = (
            lp_df[lp_df["split"] == "test"]["pinball_loss"]
            .mean()
        )

        ax.axhline(
            lp_train_mean,
            linestyle=":",
            linewidth=2,
            color="black",
            label="LP train",
        )

        ax.axhline(
            lp_test_mean,
            linestyle="-.",
            linewidth=2,
            color="black",
            label="LP test",
        )

        # -------------------------
        # Formatting
        # -------------------------
        ax.set_title(dataset_labels.get(dataset, dataset), fontsize=14)

        if use_log_scale:
            ax.set_yscale("log")

        if dataset == "abalone":
            ticks = [0.6, 0.4, 0.3, 0.25]
            ax.set_yticks(ticks)
            ax.set_yticklabels([str(t) for t in ticks])
            ax.yaxis.set_minor_locator(mticker.NullLocator())

        if dataset == "concrete":
            ticks = [2.5, 2.0, 1.7]
            ax.set_yticks(ticks)
            ax.set_yticklabels([str(t) for t in ticks])
            ax.yaxis.set_minor_locator(mticker.NullLocator())

        ax.set_xlabel("Iteration", fontsize=14)
        ax.grid(True, which="both", linewidth=0.5)

    axes[0].set_ylabel("Pinball loss", fontsize=14)
    axes[0].legend(fontsize=12)

    plt.tight_layout()

    return fig, axes

## CPS plots

In [ ]:
def plot_cps_runtime_scaling(
    df,
    *,
    metric_col="metric",
    value_col="value",
    runtime_metric="fit_seconds",
    n_col="N_SUB",
    method_col="method",
    method_label_map=None,
    pretty_methods=("LP", "IRLS", "ProxSGD"),
    figsize=(8.8, 5.8),
    save_path=None,
    dpi=600,
    return_summaries=False,
):
    """
    Recreate the CPS runtime scaling plot used in the paper.

    Assumes a long-format CPS results DataFrame with columns:
      - method
      - N_SUB
      - metric
      - value

    Runtime rows are selected using metric == 'fit_seconds'. Runtime is
    aggregated across quantile fits at each method/sample-size pair using
    the mean, with min/max shown as error bars.
    """
    if method_label_map is None:
        method_label_map = {
            "sklearn_LP": "LP",
            "lp": "LP",
            "LP": "LP",
            "IRLS": "IRLS",
            "irls": "IRLS",
            "proxSGD": "ProxSGD",
            "proxsgd": "ProxSGD",
            "ProxSGD": "ProxSGD",
        }

    color_map = {
        "LP": "C0",
        "IRLS": "C2",
        "ProxSGD": "C1",
    }

    marker_map = {
        "LP": "o",
        "IRLS": "^",
        "ProxSGD": "s",
    }

    # -------------------------------------------------
    # 1) Filter runtime rows
    # -------------------------------------------------
    runtime_df = df[df[metric_col] == runtime_metric].copy()

    runtime_df[value_col] = pd.to_numeric(runtime_df[value_col], errors="coerce")
    runtime_df[n_col] = pd.to_numeric(runtime_df[n_col], errors="coerce")

    runtime_df["method_pretty"] = runtime_df[method_col].map(method_label_map)

    runtime_df = runtime_df.dropna(
        subset=[value_col, n_col, "method_pretty"]
    )

    # -------------------------------------------------
    # 2) Aggregate across quantiles at each sample size
    # -------------------------------------------------
    runtime_summary = (
        runtime_df
        .groupby([method_col, "method_pretty", n_col], as_index=False)[value_col]
        .agg(
            runtime_mean="mean",
            runtime_min="min",
            runtime_max="max",
            n_quantiles="count",
        )
        .sort_values([method_col, n_col])
    )

    # -------------------------------------------------
    # 3) Log-log fits on aggregated means
    # -------------------------------------------------
    fit_info = {}

    for method, sub in runtime_summary.groupby(method_col):
        sub = sub.sort_values(n_col).copy()

        x_raw = sub[n_col].to_numpy(dtype=float)
        y_raw = sub["runtime_mean"].to_numpy(dtype=float)

        mask = (x_raw > 0) & (y_raw > 0)
        if mask.sum() < 2:
            continue

        x = np.log(x_raw[mask])
        y = np.log(y_raw[mask])

        slope, intercept, r_value, p_value, std_err = linregress(x, y)

        sub["fit_hat"] = np.nan
        sub.loc[mask, "fit_hat"] = np.exp(intercept) * x_raw[mask] ** slope

        pretty = sub["method_pretty"].iloc[0]

        fit_info[method] = {
            "pretty": pretty,
            "slope": slope,
            "intercept": intercept,
            "r2": r_value**2,
            "p_value": p_value,
            "std_err": std_err,
            "df": sub,
        }

    slope_summary = (
        pd.DataFrame(
            [
                {
                    "method": method,
                    "pretty": info["pretty"],
                    "loglog_slope": info["slope"],
                    "r_squared": info["r2"],
                    "p_value": info["p_value"],
                    "slope_std_err": info["std_err"],
                    "n_points": len(info["df"]),
                }
                for method, info in fit_info.items()
            ]
        )
        .sort_values("pretty")
        .reset_index(drop=True)
    )

    # -------------------------------------------------
    # 4) Plot
    # -------------------------------------------------
    fig, ax = plt.subplots(figsize=figsize)

    for pretty in pretty_methods:
        sub = runtime_summary[
            runtime_summary["method_pretty"] == pretty
        ].sort_values(n_col)

        if sub.empty:
            continue

        y = sub["runtime_mean"].to_numpy()
        yerr_lower = y - sub["runtime_min"].to_numpy()
        yerr_upper = sub["runtime_max"].to_numpy() - y

        ax.errorbar(
            sub[n_col],
            y,
            yerr=[yerr_lower, yerr_upper],
            fmt=marker_map[pretty] + "-",
            linewidth=2.2,
            markersize=6.5,
            capsize=4,
            color=color_map[pretty],
            zorder=3,
        )

    # Dashed log-log fits
    for method, info in fit_info.items():
        sub = info["df"].sort_values(n_col)
        pretty = info["pretty"]

        ax.plot(
            sub[n_col],
            sub["fit_hat"],
            linestyle="--",
            linewidth=2.0,
            color=color_map[pretty],
            alpha=0.9,
            zorder=2,
        )

    # -------------------------------------------------
    # 5) Formatting
    # -------------------------------------------------
    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel("Training Sample Size", fontsize=15)
    ax.set_ylabel("Runtime (seconds)", fontsize=15)
    ax.tick_params(axis="both", labelsize=12)

    ax.xaxis.set_major_locator(LogLocator(base=10))
    ax.yaxis.set_major_locator(LogLocator(base=10))
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.yaxis.set_major_formatter(LogFormatterMathtext())

    ax.grid(True, which="major", alpha=0.25)
    ax.grid(False, which="minor")

    method_handles = [
        Line2D(
            [0], [0],
            color=color_map["LP"],
            marker=marker_map["LP"],
            linestyle="-",
            linewidth=2.2,
            markersize=6.5,
            label="LP",
        ),
        Line2D(
            [0], [0],
            color=color_map["IRLS"],
            marker=marker_map["IRLS"],
            linestyle="-",
            linewidth=2.2,
            markersize=6.5,
            label="IRLS",
        ),
        Line2D(
            [0], [0],
            color=color_map["ProxSGD"],
            marker=marker_map["ProxSGD"],
            linestyle="-",
            linewidth=2.2,
            markersize=6.5,
            label="ProxSGD",
        ),
    ]

    style_handles = [
        Line2D(
            [0], [0],
            color="black",
            linestyle="-",
            marker="o",
            linewidth=2.2,
            markersize=6,
            label="Mean across quantiles",
        ),
        Line2D(
            [0], [0],
            color="black",
            linestyle="None",
            marker="_",
            markersize=12,
            label="Min-max across quantiles",
        ),
        Line2D(
            [0], [0],
            color="black",
            linestyle="--",
            linewidth=2.0,
            label="Log-log fit",
        ),
    ]

    legend1 = ax.legend(
        handles=method_handles,
        loc="upper left",
        frameon=True,
        fontsize=11.5,
        title="Method",
        title_fontsize=12,
    )
    ax.add_artist(legend1)

    ax.legend(
        handles=style_handles,
        loc=(0.4, 0.05),
        frameon=True,
        fontsize=11.0,
        title_fontsize=12,
    )

    annotation_lines = []

    # Match paper ordering where available
    method_order = ["sklearn_LP", "lp", "LP", "IRLS", "irls", "proxSGD", "proxsgd", "ProxSGD"]
    seen_pretty = set()

    for method in method_order:
        if method in fit_info:
            pretty = fit_info[method]["pretty"]
            if pretty in seen_pretty:
                continue
            slope = fit_info[method]["slope"]
            annotation_lines.append(f"{pretty}: slope ≈ {slope:.2f}")
            seen_pretty.add(pretty)

    ax.text(
        0.98,
        0.10,
        "\n".join(annotation_lines),
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=11,
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            alpha=0.9,
            edgecolor="0.7",
        ),
    )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")

    if return_summaries:
        return fig, ax, runtime_summary, slope_summary
    else:
        return fig, ax

In [ ]:
def plot_cps_coverage_scaling(
    df,
    *,
    n_col="N_SUB",
    metric_col="metric",
    value_col="value",
    method_col="method",
    method_styles=None,
    agg="mean",
    figsize=(8, 5.5),
    title=None,
    nominal=None,
):
    """
    Plot empirical coverage versus training sample size for CPS scaling results.

    Assumes a long-format DataFrame with columns:
      - method
      - N_SUB
      - metric
      - value
    """
    if method_styles is None:
        method_styles = {
            "sklearn_LP": {"label": "LP", "color": "C0", "linestyle": "-"},
            "lp": {"label": "LP", "color": "C0", "linestyle": "-"},
            "IRLS": {"label": "IRLS", "color": "C2", "linestyle": "-"},
            "irls": {"label": "IRLS", "color": "C2", "linestyle": "-"},
            "proxSGD": {"label": "proxSGD", "color": "C1", "linestyle": "-"},
            "proxsgd": {"label": "proxSGD", "color": "C1", "linestyle": "-"},
        }

    if agg == "mean":
        agg_fn = "mean"
    elif agg == "median":
        agg_fn = "median"
    else:
        agg_fn = agg

    coverage = df[df[metric_col] == "coverage"].copy()

    coverage[value_col] = pd.to_numeric(coverage[value_col], errors="coerce")
    coverage[n_col] = pd.to_numeric(coverage[n_col], errors="coerce")

    coverage = coverage.dropna(subset=[method_col, n_col, value_col])

    coverage_tbl = (
        coverage
        .groupby([method_col, n_col], as_index=False)[value_col]
        .agg(agg_fn)
        .sort_values([method_col, n_col])
    )

    fig, ax = plt.subplots(figsize=figsize)

    for method, style in method_styles.items():
        if method not in coverage_tbl[method_col].unique():
            continue

        sub = coverage_tbl[
            coverage_tbl[method_col] == method
        ].sort_values(n_col)

        ax.plot(
            sub[n_col],
            sub[value_col],
            marker="o",
            linewidth=2,
            color=style.get("color", None),
            linestyle=style.get("linestyle", "-"),
            label=style.get("label", str(method)),
        )

    ax.set_xscale("log")
    ax.set_xlabel("Training Sample Size", fontsize=15)
    ax.set_ylabel("Empirical coverage", fontsize=15)

    if nominal is not None:
        ax.axhline(y=nominal, color='black', linestyle='--', linewidth=1)

    if title is not None:
        ax.set_title(title, fontsize=15)

    ax.tick_params(axis="both", labelsize=12)
    ax.legend(fontsize=12)
    ax.grid(True)
    plt.tight_layout()

    return fig, ax

In [ ]:
def plot_cps_width_scaling(
    df,
    *,
    n_col="N_SUB",
    metric_col="metric",
    value_col="value",
    method_col="method",
    method_styles=None,
    agg="mean",
    figsize=(8, 5.5),
    title=None,
    ylabel="Mean interval length",
):
    """
    Plot mean interval width/length versus training sample size for CPS scaling results.

    Assumes a long-format DataFrame with columns:
      - method
      - N_SUB
      - metric
      - value
    """
    if method_styles is None:
        method_styles = {
            "sklearn_LP": {"label": "LP", "color": "C0", "linestyle": "-"},
            "lp": {"label": "LP", "color": "C0", "linestyle": "-"},
            "IRLS": {"label": "IRLS", "color": "C2", "linestyle": "-"},
            "irls": {"label": "IRLS", "color": "C2", "linestyle": "-"},
            "proxSGD": {"label": "proxSGD", "color": "C1", "linestyle": "-"},
            "proxsgd": {"label": "proxSGD", "color": "C1", "linestyle": "-"},
        }

    if agg == "mean":
        agg_fn = "mean"
    elif agg == "median":
        agg_fn = "median"
    else:
        agg_fn = agg

    width = df[df[metric_col] == "mean_width"].copy()

    width[value_col] = pd.to_numeric(width[value_col], errors="coerce")
    width[n_col] = pd.to_numeric(width[n_col], errors="coerce")

    width = width.dropna(subset=[method_col, n_col, value_col])

    width_tbl = (
        width
        .groupby([method_col, n_col], as_index=False)[value_col]
        .agg(agg_fn)
        .sort_values([method_col, n_col])
    )

    fig, ax = plt.subplots(figsize=figsize)

    for method, style in method_styles.items():
        if method not in width_tbl[method_col].unique():
            continue

        sub = width_tbl[
            width_tbl[method_col] == method
        ].sort_values(n_col)

        ax.plot(
            sub[n_col],
            sub[value_col],
            marker="o",
            linewidth=2,
            color=style.get("color", None),
            linestyle=style.get("linestyle", "-"),
            label=style.get("label", str(method)),
        )

    ax.set_xscale("log")
    ax.set_xlabel("Training Sample Size", fontsize=15)
    ax.set_ylabel(ylabel, fontsize=15)

    if title is not None:
        ax.set_title(title, fontsize=15)

    ax.tick_params(axis="both", labelsize=12)
    ax.legend(fontsize=12)
    ax.grid(True)
    plt.tight_layout()

    return fig, ax